# Load the training, validation and testing data

In [131]:
import pandas as pd
import glob

files = glob.glob("jsb_chorales/jsb_chorales/train/*.csv")
train_data = [pd.read_csv(file).values for file in files]

files = glob.glob("jsb_chorales/jsb_chorales/valid/*.csv")
valid_data = [pd.read_csv(file).values for file in files]

files = glob.glob("jsb_chorales/jsb_chorales/test/*.csv")
test_data = [pd.read_csv(file).values for file in files]


# Inspect the data

In [132]:
print(len(train_data))
print(train_data[0].shape)

print(len(valid_data))
print(valid_data[0].shape)

print(len(test_data))
print(test_data[0].shape)

229
(192, 4)
76
(196, 4)
77
(228, 4)


# Prepare data for standardization

In [145]:
import numpy as np

# Compute stats from training data only
all_train_notes = np.concatenate(train_data, axis=0)
note_mean = all_train_notes.mean(axis=0)
note_std = all_train_notes.std(axis=0)

print("Note mean:", note_mean)
print("Note std:", note_std)

def normalize(x):
    return (x - note_mean) / note_std

def denormalize(x):
    return x * note_std + note_mean

Note mean: [70.32318751 64.86695155 59.33580792 50.52422684]
Note std: [4.47415266 4.18369044 4.31237141 5.32845723]


# Preparing the data for machine learning models

In [134]:
import torch
class ChoraleDataset(torch.utils.data.Dataset):
    def __init__(self, series, window_length):
        self.series = normalize(series.astype(np.float32))
        self.window_length = window_length

    def __len__(self):
        return len(self.series) - self.window_length

    def __getitem__(self, idx):
        if(idx >= len(self)):
            raise IndexError("Index out of range")

        end = idx + self.window_length
        window = self.series[idx:end]
        target = self.series[end]
        return window, target

window_length = 98
my_dataset_full_train = torch.utils.data.ConcatDataset([ChoraleDataset(series, window_length) for series in train_data])
print(len(my_dataset_full_train))

my_dataset_full_valid = torch.utils.data.ConcatDataset([ChoraleDataset(series, window_length) for series in valid_data])
print(len(my_dataset_full_valid))

my_dataset_full_test = torch.utils.data.ConcatDataset([ChoraleDataset(series, window_length) for series in test_data])
print(len(my_dataset_full_test))


32786
10960
11354


# Create the loaders

In [135]:
train_loader = torch.utils.data.DataLoader(my_dataset_full_train, batch_size=32, shuffle=True)
valid_loader = torch.utils.data.DataLoader(my_dataset_full_valid, batch_size=32)
test_loader = torch.utils.data.DataLoader(my_dataset_full_test, batch_size=32)

# Create the RNN (GRU) model

In [136]:
import torch.nn as nn
class SimpleRNNModel(torch.nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.rnn = nn.GRU(input_size, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, X):
        outputs, last_state = self.rnn(X)
        return self.output(outputs[:, -1, :])  # Use the last output for prediction

torch.manual_seed(42)
model = SimpleRNNModel(input_size=4, hidden_size=32, output_size=4).to("cuda")

# Define the train function

In [137]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.float().to("cuda"), y_batch.float().to("cuda")
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs, patience=10, factor=0.1):
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=patience, factor=factor)
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.float().to("cuda"), y_batch.float().to("cuda")
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)

        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(metric.compute().item())
        val_metric = evaluate_tm(model, valid_loader, metric).item()
        history["valid_metrics"].append(val_metric)
        scheduler.step(val_metric)
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history

# Do the training

In [138]:
torch.manual_seed(42)
loss_fn = nn.HuberLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.003, momentum=0.9)
metric = torchmetrics.MeanAbsoluteError().to("cuda")

history = train(model, optimizer, loss_fn, metric, train_loader,
                valid_loader, n_epochs=50)

Epoch 1/50, train loss: 0.1189, train metric: 0.3472, valid metric: 0.3336
Epoch 2/50, train loss: 0.0799, train metric: 0.2525, valid metric: 0.3000
Epoch 3/50, train loss: 0.0736, train metric: 0.2286, valid metric: 0.2790
Epoch 4/50, train loss: 0.0701, train metric: 0.2144, valid metric: 0.2675
Epoch 5/50, train loss: 0.0678, train metric: 0.2055, valid metric: 0.2589
Epoch 6/50, train loss: 0.0663, train metric: 0.1992, valid metric: 0.2534
Epoch 7/50, train loss: 0.0651, train metric: 0.1946, valid metric: 0.2480
Epoch 8/50, train loss: 0.0642, train metric: 0.1908, valid metric: 0.2451
Epoch 9/50, train loss: 0.0634, train metric: 0.1879, valid metric: 0.2412
Epoch 10/50, train loss: 0.0628, train metric: 0.1853, valid metric: 0.2375
Epoch 11/50, train loss: 0.0623, train metric: 0.1832, valid metric: 0.2340
Epoch 12/50, train loss: 0.0618, train metric: 0.1809, valid metric: 0.2329
Epoch 13/50, train loss: 0.0614, train metric: 0.1795, valid metric: 0.2317
Epoch 14/50, train lo

# Evaluate on a random test entry

In [149]:
window, target = my_dataset_full_test[500]

model.eval()
with torch.no_grad():
    window = torch.from_numpy(window).float().unsqueeze(0).to("cuda")  # Add batch dimension
    prediction = model(window)
    print("Target:", denormalize(target))
    print("Prediction:", denormalize(prediction.cpu().numpy().astype(np.float32)).round())

Target: [70. 68. 65. 50.]
Prediction: [[70. 68. 64. 50.]]


# Generate Bach like music

In [147]:
n_steps = 14
generated = []

model.eval()
with torch.no_grad():
    seed = normalize(test_data[0][:window_length].astype(np.float32))
    X = torch.from_numpy(seed).float().unsqueeze(0).to("cuda")

    for _ in range(n_steps):
        output = model(X)                                   # (1, 4), normalized
        X = torch.cat([X[:, 1:, :], output.unsqueeze(1)], dim=1)
        generated.append(output.squeeze(0).cpu())

predicted_sequence = denormalize(torch.stack(generated).numpy())
print("Predicted sequence:\n", predicted_sequence.round())

Predicted sequence:
 [[65. 65. 60. 42.]
 [65. 64. 59. 42.]
 [66. 64. 59. 43.]
 [66. 63. 59. 43.]
 [66. 63. 59. 43.]
 [66. 63. 59. 44.]
 [66. 63. 59. 44.]
 [66. 63. 59. 44.]
 [66. 63. 58. 45.]
 [66. 63. 58. 45.]
 [67. 63. 58. 45.]
 [67. 63. 58. 46.]
 [67. 63. 58. 46.]
 [67. 63. 58. 46.]]
